In [30]:
import pandas as pd
import pickle
import ast

In [31]:
DISEASE = input("Disease: ")
DISEASE_FOLDER = f"../output/{DISEASE}/"
RESULT_FOLDER = DISEASE_FOLDER + "leiden_results"

In [32]:
with open(f"{RESULT_FOLDER}/result_communities_selected.pkl", "rb") as f:
    communities_selected = pickle.load(f)

In [33]:
important_terms = pd.read_csv(DISEASE_FOLDER + "important_terms.csv")

In [34]:
important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,GO_ID,Slim_IDs,Overlap (value),KEGG_ID
0,0,1001,Negative Regulation Of Cilium Assembly (GO:190...,7/14,2.565064e-04,['biological regulation'],GO_Biological_Process_2023,1.939704e-06,0.0,0.0,19.106640,2.513092e+02,TCHP;LIMK2;TBC1D30;TESK1;CDK10;EVI5L;MAP4,GO:1902018,{'GO:0065007'},0.500000,NaN
1,0,1001,Early Endosome (GO:0005769),43/307,9.160869e-08,['cellular anatomical structure'],GO_Cellular_Component_2023,9.993675e-10,0.0,0.0,3.185317,6.601220e+01,WASHC4;MGRN1;TGFBRAP1;SNX13;WASHC2C;VPS26A;SNX...,GO:0005769,{'GO:0110165'},0.140065,NaN
2,0,1001,Ubiquitin-Protein Transferase Activity (GO:000...,57/412,1.712309e-09,['catalytic activity'],GO_Molecular_Function_2023,3.057695e-12,0.0,0.0,3.171127,8.407722e+01,RNF10;TRAF3IP2;PPP1R11;UBE3C;RNF19B;UBE2Z;TRIM...,GO:0004842,{'GO:0003824'},0.138350,NaN
3,0,1001,GTPase Activator Activity (GO:0005096),29/211,3.979882e-05,['molecular function regulator activity'],GO_Molecular_Function_2023,7.817625e-07,0.0,0.0,3.084684,4.337595e+01,RABGAP1;RGS14;NPRL2;AGAP2;ARHGAP1;HACD3;IQGAP2...,GO:0005096,{'GO:0098772'},0.137441,NaN
4,0,1001,Lytic Vacuole (GO:0000323),30/223,5.531503e-05,['cellular anatomical structure'],GO_Cellular_Component_2023,8.045822e-07,0.0,0.0,3.010517,4.224642e+01,WDR45B;VPS26A;TMEM97;CD1D;ZFYVE26;SNX2;DRAM1;L...,GO:0000323,{'GO:0110165'},0.134529,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1579,14,127,Intermediate Filament (GO:0005882),12/69,1.312555e-13,['cellular anatomical structure'],GO_Cellular_Component_2023,1.640693e-14,0.0,0.0,36.276430,1.151453e+03,KRT82;KRT36;KRT79;KRT78;KRT77;KRT76;KRT75;KRT8...,GO:0005882,{'GO:0110165'},0.173913,NaN
1580,14,127,Epidermis Development (GO:0008544),12/85,2.023896e-12,['developmental process'],GO_Biological_Process_2023,2.248773e-13,0.0,0.0,28.302561,8.242617e+02,LCE2B;CASP14;SPRR2F;SPRR3;SPRR2G;KLK14;KRT32;T...,GO:0008544,{'GO:0032502'},0.141176,NaN
1581,14,127,Formation Of Cornified Envelope R-HSA-6809371,23/74,9.002650e-33,['Developmental Biology'],Reactome_2022,2.250662e-33,0.0,0.0,85.955128,6.461598e+03,SPRR2F;SPRR3;SPRR2G;SPINK6;TCHH;KLK13;KLK14;LC...,NaN,NaN,0.310811,NaN
1582,14,127,Keratinization R-HSA-6805567,118/208,1.219960e-238,['Developmental Biology'],Reactome_2022,1.016633e-239,0.0,0.0,2881.967901,1.585951e+06,KRTAP24-1;LCE1A;LIPM;KRTAP3-3;KRTAP3-2;KRTAP3-...,NaN,NaN,0.567308,NaN


# Community Category df

In [35]:
df = important_terms

# ---------------------------------------------------
# 1. Parse Category column (stringified list) -> list
# ---------------------------------------------------
df["Category"] = df["Category"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# ---------------------------------------------------
# 2. Explode: one category per row
# ---------------------------------------------------
df_exp = df.explode("Category").reset_index(drop=True)

# ---------------------------------------------------
# 3. Parse Overlap "a/b" into numerator and denominator
# ---------------------------------------------------
def parse_overlap(s):
    try:
        a, b = s.split("/")
        return int(a), int(b)
    except Exception:
        return 0, 0

overlap_nums = []
overlap_dens = []

for s in df_exp["Overlap"]:
    a, b = parse_overlap(s)
    overlap_nums.append(a)
    overlap_dens.append(b)

df_exp["Overlap_num"] = overlap_nums
df_exp["Overlap_den"] = overlap_dens

# ---------------------------------------------------
# 4. Total number of terms per community
# ---------------------------------------------------
total_terms = (
    df_exp.groupby("Community Index")["Term"]
          .nunique()
)

# ---------------------------------------------------
# 5. Aggregate per (Community Index, Category)
#    - count = # of distinct terms
#    - Gross Overlap = (sum a_i) / (sum b_i)
# ---------------------------------------------------
agg = (
    df_exp.groupby(["Community Index", "Category"])
    .agg(
        count=("Term", "nunique"),
        sum_a=("Overlap_num", "sum"),
        sum_b=("Overlap_den", "sum"),
    )
    .reset_index()
)

agg["Gross Overlap"] = agg["sum_a"] / agg["sum_b"]

# ---------------------------------------------------
# 6. Add total_terms_in_comm and frac_of_comm
# ---------------------------------------------------
agg["total_terms_in_comm"] = agg["Community Index"].map(total_terms)
agg["frac_of_comm"] = agg["count"] / agg["total_terms_in_comm"]

# ---------------------------------------------------
# 7. Final output df
# ---------------------------------------------------
community_categories = agg[[
    "Community Index",
    "Category",
    "count",
    "Gross Overlap",
    "total_terms_in_comm",
    "frac_of_comm",
]]

community_categories = community_categories.sort_values(
    by=["Community Index", "count"],
    ascending=[True, False]
).reset_index(drop=True)

community_categories


,Community Index,Category,count,Gross Overlap,total_terms_in_comm,frac_of_comm
0,0,cellular process,16,0.150437,46,0.347826
1,0,biological regulation,9,0.132523,46,0.195652
2,0,catalytic activity,7,0.147831,46,0.152174
3,0,localization,6,0.156546,46,0.130435
4,0,binding,5,0.168453,46,0.108696
...,...,...,...,...,...,...
102,14,Developmental Biology,3,0.191144,9,0.333333
103,14,cellular anatomical structure,3,0.221477,9,0.333333
104,14,Infectious disease: bacterial,1,0.136842,9,0.111111
105,14,cellular process,1,0.367647,9,0.111111


In [36]:
community_categories_filtered = (
    community_categories.sort_values(["Community Index", "count"], ascending=[True, False])
          .groupby("Community Index")
          .head(5)
          .reset_index(drop=True)
)

In [37]:
community_categories_filtered

,Community Index,Category,count,Gross Overlap,total_terms_in_comm,frac_of_comm
0,0,cellular process,16,0.150437,46,0.347826
1,0,biological regulation,9,0.132523,46,0.195652
2,0,catalytic activity,7,0.147831,46,0.152174
3,0,localization,6,0.156546,46,0.130435
4,0,binding,5,0.168453,46,0.108696
5,1,Infectious disease: viral,1,0.150602,2,0.500000
6,1,transcription regulator activity,1,0.131687,2,0.500000
7,2,biological regulation,667,0.254933,1402,0.475749
8,2,cellular process,235,0.218558,1402,0.167618
9,2,response to stimulus,99,0.244812,1402,0.070613


In [38]:
from reportlab.lib.pagesizes import letter, landscape
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle
from reportlab.lib import colors

def df_to_pdf(df, filename="output.pdf"):
    pdf = SimpleDocTemplate(
        filename,
        pagesize=landscape(letter)
    )

    # Convert DF to 2D list
    data = [df.columns.tolist()] + df.values.tolist()

    # Create table
    table = Table(data)

    # Styling
    style = TableStyle([
        ("BACKGROUND", (0,0), (-1,0), colors.lightgrey),
        ("TEXTCOLOR", (0,0), (-1,0), colors.black),
        ("ALIGN", (0,0), (-1,-1), "CENTER"),
        ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"),
        ("FONTSIZE", (0,0), (-1,-1), 8),
        ("BOTTOMPADDING", (0,0), (-1,0), 8),
        ("GRID", (0,0), (-1,-1), 0.25, colors.black),
    ])
    table.setStyle(style)

    elements = [table]
    pdf.build(elements)
    print(f"PDF saved to {filename}")


In [39]:
df_to_pdf(community_categories_filtered, f"../../Graphs/community_categories_filtered_{DISEASE}.pdf")

PDF saved to ../../Graphs/community_categories_filtered_NONE.pdf


# Community Count Dict

In [40]:
category_count_by_comm = {}

for _, row in community_categories.iterrows():
    comm = row["Community Index"]
    cat = row["Category"]
    count = row["count"]
    gross = row["Gross Overlap"]

    if comm not in category_count_by_comm:
        category_count_by_comm[comm] = {}

    # insertion happens in sorted order because community_categories is sorted
    category_count_by_comm[comm][cat] = (count, gross)


In [41]:
for i in range(len(communities_selected)):
    if i not in category_count_by_comm.keys():
        category_count_by_comm[i] = {}

In [42]:
category_count_by_comm

{0: {'cellular process': (16, 0.15043731778425656),
  'biological regulation': (9, 0.13252346769740475),
  'catalytic activity': (7, 0.14783139890042762),
  'localization': (6, 0.15654648956356737),
  'binding': (5, 0.16845329249617153),
  'cellular anatomical structure': (2, 0.13773584905660377),
  'molecular function regulator activity': (2, 0.12440944881889764),
  'protein-containing complex': (2, 0.15789473684210525),
  'Signal Transduction': (1, 0.12018140589569161),
  'developmental process': (1, 0.10392609699769054)},
 1: {'Infectious disease: viral': (1, 0.15060240963855423),
  'transcription regulator activity': (1, 0.13168724279835392)},
 2: {'biological regulation': (667, 0.2549328571178458),
  'cellular process': (235, 0.21855849263697064),
  'response to stimulus': (99, 0.2448116325181758),
  'binding': (79, 0.17108058348675825),
  'developmental process': (74, 0.27291791491911327),
  'Signal Transduction': (57, 0.17845292409788469),
  'cellular anatomical structure': (40,

In [43]:
with open(f"{DISEASE_FOLDER}/category_count_by_comm.pkl", "wb") as f:
    pickle.dump(category_count_by_comm, f)